# Quickstart: a dataset from one sentence

Describe the data you need in plain English and get back related tables as CSV files. This is the shortest path from nothing to usable test data.

**Uses:** 2,000 records, no AI-generated fields. API reference: [generaterealdata.com/docs/api/](https://www.generaterealdata.com/docs/api/?ref=github).

## Setup

You need an API key (PRO, ULTRA or Enterprise plan): sign in at [generaterealdata.com](https://www.generaterealdata.com?ref=github), open the dashboard and click **Generate API key**. Keep it out of the notebook: put it in the `GRD_API_KEY` environment variable, or in Colab's **Secrets** panel under the same name.

In [1]:
import os
import time
from pathlib import Path

import requests

API = "https://www.generaterealdata.com/api/v1"
KEY = os.environ.get("GRD_API_KEY")
if not KEY:                                   # Google Colab: read it from the Secrets panel
    from google.colab import userdata
    KEY = userdata.get("GRD_API_KEY")
HEADERS = {"Authorization": f"Bearer {KEY}"}
OUT = Path("downloads")
OUT.mkdir(exist_ok=True)


def api(method, path, **kwargs):
    """One API call. Waits and tries again if the API says to slow down or another dataset is still running."""
    while True:
        r = requests.request(method, API + path, headers=HEADERS, timeout=90, **kwargs)
        busy = r.status_code == 409 and r.json().get("error", {}).get("code") == "run_in_progress"
        if r.status_code == 429 or busy:
            time.sleep(int(r.headers.get("Retry-After", 15)))
            continue
        data = r.json()
        if r.status_code >= 400:
            e = data["error"]
            raise RuntimeError(f"{e['code']}: {e['message']} {e.get('hint', '')}")
        return data


def generate(spec):
    """Start a dataset and wait until it is ready. Returns its final status."""
    ds = api("POST", "/datasets", json=spec)
    while ds["status"] not in ("ready", "failed"):
        time.sleep(5)
        ds = api("GET", f"/datasets/{ds['id']}")
    if ds["status"] == "failed":
        raise RuntimeError(ds["error"]["message"])
    return ds


def download(ds, filename, file=None):
    """Save a ready dataset's file. file="db" fetches the SQLite file of an SQL dataset."""
    path = OUT / filename
    url = f"{API}/datasets/{ds['id']}/download"
    with requests.get(url, params={"file": file} if file else None, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return path


def summary(ds):
    """The facts worth reading from a dataset's status."""
    keys = ("status", "format", "rows_generated", "rows_per_table", "credits_charged", "ai_rows_charged", "schema_id", "seed")
    return {k: ds[k] for k in keys if k in ds}

## 1. Describe the data

`mode: "prompt"` lets the service design the tables, columns and relationships from your description. Without `use_ai`, every value comes from a built-in generator, so no AI-generated fields are used.

In [2]:
spec = {
    "mode": "prompt",
    "description": ("A fitness studio chain with members, membership plans (Basic, Plus, Premium), classes (Yoga, Spin, Pilates, "
                    "HIIT, Boxing, Zumba) and class bookings. Members live in New York, Chicago, Austin, Seattle and Denver. "
                    "Bookings run from January 2025 to September 2026 and are attended, cancelled or no-show."),
    "rows": 2000,
    "format": "csv",
    "seed": 11,
}
check = api("POST", "/datasets", json={**spec, "dry_run": True})
print(check["message"])
print("Records needed:", check["credits_required"])

Everything checks out. Send the same request without "dry_run" to generate.
Records needed: 2000


`dry_run` validates the request and tells you what it will use, without generating or charging anything.

## 2. Generate and download

In [3]:
ds = generate(spec)
summary(ds)

{'status': 'ready',
 'format': 'csv',
 'rows_generated': 2000,
 'rows_per_table': {'locations': 5,
  'bookings': 1600,
  'members': 386,
  'classes': 6,
  'plans': 3},
 'credits_charged': 2000,
 'ai_rows_charged': 0,
 'schema_id': 'sch_SH0BM5SlnWOXxjGM',
 'seed': 11}

In [4]:
import zipfile

path = download(ds, "fitness_studio.zip")
archive = zipfile.ZipFile(path)
[(name, f"{archive.getinfo(name).file_size:,} bytes") for name in archive.namelist()]

[('classes.csv', '153 bytes'),
 ('locations.csv', '93 bytes'),
 ('plans.csv', '69 bytes'),
 ('members.csv', '33,445 bytes'),
 ('bookings.csv', '88,513 bytes')]

## 3. Look at the tables

Several tables arrive as one zip with a CSV per table.

In [5]:
import pandas as pd

tables = {name.rsplit("/", 1)[-1].removesuffix(".csv"): pd.read_csv(archive.open(name)) for name in archive.namelist() if name.endswith(".csv")}
for name, df in tables.items():
    print(f"\n{name}: {len(df):,} rows")
    display(df.head(3))


classes: 6 rows


,class_id,class_name,duration_minutes,intensity
0,1,Yoga,60,Low
1,2,Spin,45,High
2,3,Pilates,50,Medium



locations: 5 rows


,location_id,city,state
0,1,New York,NY
1,2,Chicago,IL
2,3,Austin,TX



plans: 3 rows


,plan_id,plan_name,monthly_fee
0,1,Basic,50
1,2,Plus,80
2,3,Premium,120



members: 386 rows


,member_id,first_name,last_name,email,phone,location_id,plan_id,join_date,status
0,MEM-1000,Shaun,Garcia,patriciapowell@example.com,594.390.2431x48530,4,1,2024-03-07,Active
1,MEM-1001,Anita,Simon,fcastaneda@example.com,783-773-0682,5,2,2023-05-10,Active
2,MEM-1002,Sandra,Martinez,donna17@example.org,616-326-3536,4,1,2023-03-07,Active



bookings: 1,600 rows


,booking_id,member_id,location_id,class_id,booking_date,status,fee_charged
0,BKG-10000,MEM-1099,4,2,2026-06-01 08:37:34,Cancelled,10
1,BKG-10001,MEM-1046,3,5,2025-10-18 13:53:46,Attended,0
2,BKG-10002,MEM-1041,3,1,2026-08-12 16:32:56,Attended,0


## 4. The tables are related

Every column that points at another table holds a value that exists there. Here is that check for each `*_id` column that is another table's first column.

In [6]:
first_column = {df.columns[0]: name for name, df in tables.items()}
for name, df in tables.items():
    for column in df.columns[1:]:
        parent = first_column.get(column)
        if parent and parent != name:
            missing = (~df[column].dropna().isin(tables[parent][column])).sum()
            print(f"{name}.{column} -> {parent}.{column}: {missing} values without a match")

members.location_id -> locations.location_id: 0 values without a match
members.plan_id -> plans.plan_id: 0 values without a match
bookings.member_id -> members.member_id: 0 values without a match
bookings.location_id -> locations.location_id: 0 values without a match
bookings.class_id -> classes.class_id: 0 values without a match


## Next

- The response includes a `schema_id`. Send it with `mode: "schema"` to get the same structure again without the design step, and add a `seed` for identical data ([notebook 05](05_same_data_every_time_with_seeds.ipynb)).
- Have a database schema already? Use your `CREATE TABLE` statements instead of a description ([notebook 02](02_generate_from_your_sql_schema.ipynb)).
- Want reviews, notes or other free text that fits each row? Add `"use_ai": true`; each AI-written value uses one AI-generated field.